# Embeddings

## Character Embeddings

Let's learn the character embeddings from Mary Shelley's Frankenstein (or any of your favorite [Project Gutenberg](https://www.gutenberg.org/) texts). We train a Neural Net to predict the next token (character) given the previous character.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import string
from collections import Counter

import matplotlib.pyplot as plt

torch.manual_seed(0)

file_name = "Frankenstein.txt"

with open(file_name, 'r') as f:
    txt = f.read()

def preprocess(t):
    t = t.lower()
    t = " ".join(t.split())

    # remove punctuation and special characters
    for s in string.punctuation + "æèéêô—‘’“”ù":
        t = t.replace(s, "")
    return t

In [ ]:
data = preprocess(txt)
print(sorted(set(data)))
print(f"Number of characters: {len(set(data))}")

In [ ]:
words = data.lower().split()
word_counts = Counter(words)

for word, count in word_counts.most_common(20):
    print(f"{word:15} {count}")

We've cleaned our data and found that there are 37 characters. Let's create our dataset. Given a token $x_i$, we want to predict what the token at $x_{i+1}$ should be.

Our input will be the text sequence and the output will be the text sequence offset by one.

For the sequence '12345', we would want the NN to predict '2' when given '1', predict '3' when given '2', predict '4' when given '3', and predict '5' when given '4'. So the input sequence looks like '1234' and the targeted output is '2345'.


In [ ]:
def create_inp_out(data):
    inp = data[:-1]
    out = data[1:]
    return inp, out

# short example
sample_data = "12345"
sample_input, sample_target = create_inp_out(sample_data)
print(sample_input)
print(sample_target)

In [ ]:
# creating the input and target for Frankenstein characters
inp, target = create_inp_out(data)

We need to represent our data as a one-hot encoding. Since there are 37 characters, there are 37 classes for prediction. The first column (index 0) will be the character ' ', the second column (index 1) will be the character '1', etc.

In [ ]:
char2idx = {k: v for v, k in enumerate(sorted(set(data)))}
idx2char = {v: k for k, v in char2idx.items()}

vocab_size = len(char2idx)

print(char2idx)
print(idx2char)

In [ ]:
def convert_to_one_hot(txt, char2idx, vocab_size):
    res = [char2idx[ch] for ch in txt]
    res = torch.tensor(res)
    return torch.nn.functional.one_hot(res, num_classes=vocab_size).float()

def convert_to_indices(txt, char2idx):
    res = [char2idx[ch] for ch in txt]
    return torch.tensor(res)

X = convert_to_one_hot(inp, char2idx, vocab_size)
y = convert_to_one_hot(target, char2idx, vocab_size)

# small example of OHE for '12345' example
sample_X = convert_to_one_hot(sample_input, char2idx, vocab_size)
sample_y = convert_to_indices(sample_target, char2idx)
print(sample_X)
print(sample_y)

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        return self.X[index], self.y[index]

def get_dataloader(X, y, batch_size=256):
    dataset = Dataset(X, y)
    params = {'batch_size': batch_size, 'shuffle': False, 'num_workers': 0}
    return torch.utils.data.DataLoader(dataset, **params)

We define a simple feed-forward network:
- Input: one-hot vector
- Embedding layer: projects to 2D
- Output layer: projects back to vocabulary size

In [ ]:
class Net(nn.Module):
    def __init__(self, vocab_size):
        super(Net, self).__init__()
        self.fc_embedding = nn.Linear(vocab_size, 2)
        self.fc2 = nn.Linear(2, vocab_size)
    def forward(self, x):
        x = self.fc_embedding(x)
        x = self.fc2(x)
        # to predict a likelihood of the next character, we SHOULD include Softmax. Howeer, torch's CrossEntropyLoss() already does a Softmax, so don't do it here!
        return x

We can visualize the 2D embeddings of each character before and after training.

In [ ]:
def plot_embeddings(model, idx2char):
    plt.figure(figsize=(8,8))
    plt.axis("equal")
    vocab_size = len(idx2char)
    res = model.fc_embedding(torch.eye(vocab_size))
    res = res.cpu().detach().numpy()
    plt.scatter(res[:, 0], res[:, 1], c='white')
    for idx, char in idx2char.items():
        if char == " ":
            char = "*"
        plt.text(res[idx, 0], res[idx, 1], char)
    plt.show()

In [ ]:
def train(model, dataloader, lr=0.01, epochs=50):
    # this loss criterion ALSO does softmax across the outputs
    loss_criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for e in range(epochs):
        running_loss = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs, labels

            optimizer.zero_grad()
            logits = model(inputs)

            loss = loss_criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        epoch_loss = running_loss / len(dataloader)
        print(f"Epoch {e+1:3d} | Average Loss: {epoch_loss:.4f}")


In [ ]:
inp, target = create_inp_out(data)

X = convert_to_one_hot(inp, char2idx, vocab_size)
y = convert_to_indices(target, char2idx)
full_loader = get_dataloader(X, y, batch_size=256)

print("\n=== Training on full dataset ===")
NN_full = Net(vocab_size)
plot_embeddings(NN_full, idx2char)

In [ ]:
train(NN_full, full_loader, epochs=100, lr=0.01)

In [ ]:
plot_embeddings(NN_full, idx2char)

#### Question:

What do you notice about the character embeddings before and after training?

In [ ]:
# cosine_similarity returns the cosine of the angle between two vectors
# u and v
def cosine_similarity(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

def most_similar_character(char, embeddings, char2idx, idx2char, k=5):
    i = char2idx[char]
    sims = []

    for j in range(len(idx2char)):
        if i == j:
            continue

        sim = cosine_similarity(embeddings[i], embeddings[j])
        sims.append((idx2char[j], sim))

    sims.sort(key=lambda x: x[1], reverse=True)

    print(f"Most similar to '{char}':")
    for c, s in sims[:k]:
        print(f"  {c:>3}: {s:.3f}")

In [ ]:
embeddings = NN_full.fc_embedding.weight.detach().numpy().T

In [ ]:
most_similar_character('o', embeddings, char2idx, idx2char)